In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as cp
from scipy.interpolate import griddata
from matplotlib.colors import LogNorm
from matplotlib.gridspec import GridSpec
from scipy.interpolate import interp1d
from FallbackGen import FallbackGen
from TDECalculator import TDECalculator
import gc

In [2]:
MBH = 1e6
Rp = 23
a = 0.0
N = 5000
E = 1.0
Q = 0.0
orbit="rel"

In [3]:
mass_sch = TDECalculator('MAMS1Msun', orbit, MBH, Rp, a, N=N)

In [4]:
sample_sch = mass_sch.rel_whole_star_sample()

In [5]:
radii_sch = sample_sch['rr']

rtde = mass_sch.R_TDE
Lz = mass_sch.mom_kerr_analytic(Rp, a)
E = 1.0
Q = 0

radii = np.where(
    radii_sch <= 0.5,
    rtde - radii_sch * mass_sch.Rstar,
    rtde + radii_sch * mass_sch.Rstar
)

deltaE = mass_sch.Rstar / mass_sch.Rp**2 

i = int(np.random.uniform(0, 1132))
j = int(np.random.uniform(0, 90000))
dE = sample_sch['dEnergy_random'] * deltaE 
dLz = mass_sch.dLz_random
dQ = mass_sch.dQ_random
mass_ratio = mass_sch.mass_ratio

In [6]:
dT_sch = FallbackGen(mass_ratio, a, radii, Rp, E, Lz, Q, dE, dLz, dQ, N)
np.save('dT_sch.npy', dT_sch.dTs)
gc.collect()

Computing radial periods for 101,880,000 particles ...
  E  range: [0.998878, 1.001122]
  Q  range: [-1.139e-04, 1.139e-04]
  chunk_size = 50,000
  Finding roots (chunked eigensolver) ...
  roots chunk 10119/10119 (100%)
  Bound: 50,591,365 / 101,880,000
  Valid roots: 50,591,365
  Valid Lambda_r: 50,591,365
  Quadrature: 1012 chunks ...
    chunk 1012/1012  (100%)
  Successful T_r: 50,591,365 / 101,880,000


20

In [ ]:
def make_plot_dicts(whole_star_sample, dT, delta):
    dE_rand = whole_star_sample['dEnergy_random']
    dT_rand = dT / delta
    dMass   = whole_star_sample['dMass']

    bins_E = np.linspace(-2, 2, 1000)
    bins_T = np.logspace(0, 6, 1000)

    # dE: use all particles with finite energy and mass
    valid_E = np.isfinite(dE_rand)
    hist_E, edges_E = np.histogram(dE_rand[valid_E], bins=bins_E,
                                   weights=dMass[valid_E], density=False)

    # dT: only particles with valid finite period within bin range
    valid_T = (np.isfinite(dT_rand) & np.isfinite(dE_rand)
               & (dT_rand >= 1.0) & (dT_rand <= 1e6))
    hist_T, edges_T = np.histogram(dT_rand[valid_T], bins=bins_T,
                                   weights=dMass[valid_T], density=False)
    
    hist_E /= np.diff(bins_E)
    hist_T /= np.diff(bins_T)

    return {"x": 0.5*(edges_E[:-1]+edges_E[1:]), "y": hist_E}, \
           {"x": 0.5*(edges_T[:-1]+edges_T[1:]), "y": hist_T}

In [8]:
def energy_plots(whole_star_sample): 
    # unpack
    dE_rand   = whole_star_sample['dEnergy_random']      # shape (nr-1, N_Omega)
    dE_unp    = whole_star_sample['dEnergy_unperturbed']
    dT_rand   = whole_star_sample['dT_random']
    dT_unp    = whole_star_sample['dT_unperturbed']
    dMass     = whole_star_sample['dMass']               # same shape

    # set up figure
    # fig, (axE, axT) = plt.subplots(1, 2, figsize=(12,5))

    # histograms in E
    bins_E = np.linspace(-2, 2, 200)
    # axE.hist(dE_rand.flatten(),
    #         weights=dMass.flatten(),
    #         bins=bins_E,
    #         density=True,
    #         alpha=0.5)
    # axE.set_yscale('log')
    # axE.set_xlabel(r'$ E / \Delta E$')
    # axE.set_ylabel(r'$(\mathrm{d}M/\mathrm{d}E) / (M_\star / \Delta E)$')
    # axE.set_xlim(-1,1)
    # axE.set_ylim(1e-5)
 
    # histograms in T
    bins_T = np.logspace(0, 6, 200)
#     axT.hist(dT_rand.flatten(),
#             weights=dMass.flatten(),
#             bins=bins_T,
#             density=True,
#             alpha=0.5)
#     # overplot T^{-5/3} reference
#     axT.plot(bins_T, 10*bins_T**(-5/3), 'k--', label=r'$T^{-5/3}$')
#     axT.set_xscale('log')
#     axT.set_yscale('log')
#     axT.set_xlabel(r'$T / \Delta T$')
#     axT.set_ylabel(r'$(\mathrm{d}M/\mathrm{d}T) / (M_\star / \Delta T)$')
#     axT.set_xlim(1,2000)
#     axT.set_ylim(2e-5,0.1)

#     plt.tight_layout()
#     plt.show()
    
    hist_vals_E, bin_edges_E = np.histogram(
        dE_rand.flatten(),
        bins=bins_E,
        weights=dMass.flatten(),
        density=True
        )
    
    hist_vals_T, bin_edges_T = np.histogram(
        dT_rand.flatten(),
        bins=bins_T,
        weights=dMass.flatten(),
        density=True
        )
    
    bin_centers_E = 0.5 * (bin_edges_E[:-1] + bin_edges_E[1:]) # Save the curve for later
    bin_centers_T = 0.5 * (bin_edges_T[:-1] + bin_edges_T[1:]) # Save the curve for later
    
    curve_E = {
        "x": bin_centers_E,
        "y": hist_vals_E
        }
    
    curve_T = {
        "x": bin_centers_T,
        "y": hist_vals_T
        }
    
    return curve_E, curve_T

In [9]:
DeltaE = mass_sch.Rstar / mass_sch.Rp**2
DeltaT = 1 / DeltaE**1.5
n_ex = TDECalculator('MAMS1Msun', Rp=18, a=0.0, N=N)

In [10]:
whole_star_sample4 = n_ex.whole_star_sample()

In [11]:
n_E, n_T = energy_plots(whole_star_sample4)

In [12]:
rel_sch_E, rel_sch_T = make_plot_dicts(sample_sch, dT_sch.dTs, DeltaT)

In [13]:
import json

adden = "m1_rp23"

with open(f"Fallback_Data/rel_sch_e_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in rel_sch_E.items()}, f)
with open(f"Fallback_Data/rel_sch_t_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in rel_sch_T.items()}, f)

adden = "m1_rp18"

with open(f"Fallback_Data/n_e_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in n_E.items()}, f)
with open(f"Fallback_Data/n_t_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in n_T.items()}, f)
